## 1. Install Library

In [ ]:
!pip install torch transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

## 2. Persiapan Data

In [ ]:
import pandas as pd
from datasets import Dataset

# Contoh DataFrame
data = {
    "product": [
        "Smartphone dengan kamera 108MP",
        "Baju kaos katun warna hitam",
        "Buku novel karya Tere Liye",
        "Laptop gaming dengan RAM 16GB",
        "Sepatu lari Nike",
        "Buku resep masakan"
    ],
    "category": [
        "Electronics",
        "Fashion",
        "Books",
        "Electronics",
        "Fashion",
        "Books"
    ]
}

df = pd.DataFrame(data)

# Konversi DataFrame ke Dataset Hugging Face
dataset = Dataset.from_pandas(df)

# Pisahkan dataset menjadi train dan test
dataset = dataset.train_test_split(test_size=0.2)

## 3. Tokenisasi Data

In [ ]:
from transformers import AutoTokenizer

# Pilih model google/gemma-3-1b-pt
model_name = "google/gemma-2b"
token = ""
tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=token)

# Jika tokenizer tidak memiliki pad_token, tambahkan atau gunakan eos_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Gunakan eos_token sebagai pad_token

# Tokenisasi dataset
def tokenize_function(examples):
    return tokenizer(examples["product"], padding="max_length", truncation=True, max_length=32)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.11/dist-packages/transformers/models/auto/tokenization_auto.py:823: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/33.6k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [ ]:
# !pip install --upgrade transformers torch

In [ ]:
# !git clone https://github.com/huggingface/transformers.git

In [ ]:
# !cd transformers && pip install .

## 4. Persiapan Model

In [ ]:
from transformers import AutoModelForSequenceClassification

# Pilih model dengan lapisan klasifikasi
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, use_auth_token=token,
    num_labels=len(df["category"].unique())  # Jumlah kategori unik
)

/usr/local/lib/python3.11/dist-packages/transformers/models/auto/auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of GemmaForSequenceClassification were not initialized from the model checkpoint at google/gemma-2b and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 5. Fine-Tuning Model

In [ ]:
from transformers import Trainer, TrainingArguments
import torch

# Pastikan model berada di perangkat yang benar (GPU atau CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Persiapan label
label_list = df["category"].unique().tolist()
label_to_id = {label: idx for idx, label in enumerate(label_list)}

def map_labels(examples):
    examples["label"] = label_to_id[examples["category"]]
    return examples

tokenized_datasets = tokenized_datasets.map(map_labels)

# Argumen pelatihan
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,  # Kurangi dari 16 ke 8 atau lebih kecil
    per_device_eval_batch_size=4,   # Kurangi dari 16 ke 8 atau lebih kecil'
    gradient_accumulation_steps=4,  # Akumulasi gradien selama 4 langkah
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    report_to="none",
)

# Inisialisasi Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
)

# Mulai fine-tuning
trainer.train()

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-8-8522d84c26a2>:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,No log,1.107349
2,No log,0.811823
3,No log,0.653173


TrainOutput(global_step=3, training_loss=0.405990997950236, metrics={'train_runtime': 236.5064, 'train_samples_per_second': 0.051, 'train_steps_per_second': 0.013, 'total_flos': 4566275850240.0, 'train_loss': 0.405990997950236, 'epoch': 3.0})

## 6. Evaluasi Model

In [ ]:
# Evaluasi model
eval_results = trainer.evaluate()
print(f"Evaluasi hasil: {eval_results}")

Evaluasi hasil: {'eval_loss': 0.6531732082366943, 'eval_runtime': 2.8468, 'eval_samples_per_second': 0.703, 'eval_steps_per_second': 0.351, 'epoch': 3.0}


## 7. Menggunakan Model untuk Prediksi

In [ ]:
# Fungsi untuk prediksi
def predict_category(product_text):
    # Tokenisasi input
    inputs = tokenizer(product_text, return_tensors="pt", padding=True, truncation=True, max_length=64)

    # Pindahkan input ke perangkat yang sama dengan model
    inputs = {key: value.to(device) for key, value in inputs.items()}

    # Lakukan prediksi
    with torch.no_grad():  # Nonaktifkan perhitungan gradien
        outputs = model(**inputs)

    # Ambil prediksi kelas
    predicted_class_id = outputs.logits.argmax().item()
    return label_list[predicted_class_id]

# Test prediksi
test_products = [
    "Smartphone dengan baterai 5000mAh",
    "Jaket kulit pria",
    "Buku panduan programming Python"
]

for product in test_products:
    predicted_category = predict_category(product)
    print(f"Product: {product}")
    print(f"Predicted Category: {predicted_category}")
    print("-" * 50)

Product: Smartphone dengan baterai 5000mAh
Predicted Category: Electronics
--------------------------------------------------
Product: Jaket kulit pria
Predicted Category: Fashion
--------------------------------------------------
Product: Buku panduan programming Python
Predicted Category: Books
--------------------------------------------------


In [ ]:
import torch
import torch.nn.functional as F

def predict_category(product_text):
    # Tokenisasi input
    inputs = tokenizer(product_text, return_tensors="pt", padding=True, truncation=True, max_length=64)

    # Pindahkan input ke perangkat yang sama dengan model
    inputs = {key: value.to(device) for key, value in inputs.items()}

    # Lakukan prediksi
    with torch.no_grad():  # Nonaktifkan perhitungan gradien
        outputs = model(**inputs)

    # Ambil logits (output mentah dari model)
    logits = outputs.logits

    # Hitung probabilitas menggunakan softmax
    probabilities = F.softmax(logits, dim=-1)

    # Ambil prediksi kelas dan probabilitasnya
    predicted_class_id = logits.argmax().item()
    predicted_probability = probabilities[0][predicted_class_id].item()

    # Ambil probabilitas untuk semua kelas
    all_probabilities = {label: prob.item() for label, prob in zip(label_list, probabilities[0])}

    return {
        "predicted_category": label_list[predicted_class_id],
        "predicted_probability": predicted_probability,
        "all_probabilities": all_probabilities
    }

# Test prediksi
test_products = [
    "Smartphone dengan baterai 5000mAh",
    "Jaket kulit pria",
    "Buku panduan programming Python"
]

for product in test_products:
    result = predict_category(product)
    print(f"Product: {product}")
    print(f"Predicted Category: {result['predicted_category']}")
    print(f"Predicted Probability: {result['predicted_probability']:.4f}")
    print("All Probabilities:")
    for label, prob in result["all_probabilities"].items():
        print(f"  {label}: {prob:.4f}")
    print("-" * 50)

Product: Smartphone dengan baterai 5000mAh
Predicted Category: Electronics
Predicted Probability: 0.5407
All Probabilities:
  Electronics: 0.5407
  Fashion: 0.2875
  Books: 0.1718
--------------------------------------------------
Product: Jaket kulit pria
Predicted Category: Fashion
Predicted Probability: 0.6482
All Probabilities:
  Electronics: 0.0091
  Fashion: 0.6482
  Books: 0.3427
--------------------------------------------------
Product: Buku panduan programming Python
Predicted Category: Books
Predicted Probability: 0.5706
All Probabilities:
  Electronics: 0.3512
  Fashion: 0.0781
  Books: 0.5706
--------------------------------------------------
